# Murphy 1 · Chapter 3 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M3). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 3.1 · Covariance by hand, and a trap  
*Section §3.1 · code*

(a) For x = [1,2,3,4,5] and y = [2,4,5,4,5], compute the covariance and correlation by hand in NumPy and check against np.cov / np.corrcoef. (b) Let x be 10,000 standard normals and y = x². Show the correlation is ≈ 0 even though y is a function of x.

<details><summary><b>Hint</b></summary>

cov = mean((x − x̄)(y − ȳ)); np.cov uses n − 1 in the denominator by default.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
x = np.array([1, 2, 3, 4, 5.]); y = np.array([2, 4, 5, 4, 5.])
cov = ...  # TODO: average product of deviations (n-1)
corr = ...  # TODO: rescale to -1..1
print("by hand :", round(cov, 4), round(corr, 4))
print("numpy   :", round(np.cov(x, y)[0, 1], 4), round(np.corrcoef(x, y)[0, 1], 4))

rng = np.random.default_rng(3)
a = rng.normal(size=10000); b = a ** 2
print("corr(x, x^2) =", round(np.corrcoef(a, b)[0, 1], 4))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
x = np.array([1, 2, 3, 4, 5.]); y = np.array([2, 4, 5, 4, 5.])
cov = np.sum((x - x.mean()) * (y - y.mean())) / (len(x) - 1)  # average product of deviations (n-1)
corr = cov / (x.std(ddof=1) * y.std(ddof=1))  # rescale to -1..1
print("by hand :", round(cov, 4), round(corr, 4))
print("numpy   :", round(np.cov(x, y)[0, 1], 4), round(np.corrcoef(x, y)[0, 1], 4))

rng = np.random.default_rng(3)
a = rng.normal(size=10000); b = a ** 2
print("corr(x, x^2) =", round(np.corrcoef(a, b)[0, 1], 4))

<details><summary><b>Answer and explanation</b></summary>

(a) Covariance (n − 1 version) = 1.5, correlation ≈ 0.775. (b) Correlation ≈ 0, because for every positive x there's a matching negative x with the same y, so the linear trend cancels. Correlation only measures straight-line relationships; zero correlation does not mean independence (Murphy §3.1.3).

Expected output:
```
by hand : 1.5 0.7746
numpy   : 1.5 0.7746
corr(x, x^2) = -0.0034
```

</details>

## 3.2 · Sample a tilted Gaussian with Cholesky  
*Section §3.2 · code*

Draw 50,000 samples from N(μ, Σ) with μ = [1, −2] and Σ = [[2, 1.2], [1.2, 1]] using only standard normal draws and the Cholesky factor L (Σ = L Lᵀ). Check the sample mean and covariance.

<details><summary><b>Hint</b></summary>

If z ~ N(0, I) then μ + L z ~ N(μ, L Lᵀ).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(4)
mu = np.array([1., -2.]); Sigma = np.array([[2., 1.2], [1.2, 1.]])
L = np.linalg.cholesky(Sigma)
z = rng.normal(size=(50000, 2))
x = ...  # TODO: stretch and tilt the standard noise, then shift
print("L =", L.round(3).tolist())
print("sample mean:", x.mean(0).round(3))
print("sample cov :", np.cov(x.T).round(3).tolist())

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(4)
mu = np.array([1., -2.]); Sigma = np.array([[2., 1.2], [1.2, 1.]])
L = np.linalg.cholesky(Sigma)
z = rng.normal(size=(50000, 2))
x = mu + z @ L.T  # stretch and tilt the standard noise, then shift
print("L =", L.round(3).tolist())
print("sample mean:", x.mean(0).round(3))
print("sample cov :", np.cov(x.T).round(3).tolist())

<details><summary><b>Answer and explanation</b></summary>

Because Cov(Lz) = L Cov(z) Lᵀ = L I Lᵀ = Σ, stretching standard noise by L produces exactly the covariance you want. The sample covariance comes out within about 0.01–0.02 of Σ.

Expected output:
```
L = [[1.414, 0.0], [0.849, 0.529]]
sample mean: [ 1.003 -1.996]
sample cov : [[1.989, 1.194], [1.194, 0.998]]
```

</details>

## 3.3 · Condition a 2-D Gaussian  
*Section §3.2 · derive*

Heights and weights are jointly Gaussian with μ_h = 170 cm, σ_h = 10, μ_w = 70 kg, σ_w = 12, correlation ρ = 0.6. What are the mean and standard deviation of weight for someone 190 cm tall?

<details><summary><b>Hint</b></summary>

E[w|h] = μ_w + ρ(σ_w/σ_h)(h − μ_h);  Var[w|h] = σ_w²(1 − ρ²).

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

E[w | h=190] = 70 + 0.6 × (12/10) × 20 = 70 + 14.4 = 84.4 kg. Var = 144 × (1 − 0.36) = 92.16, so σ ≈ 9.6 kg. Knowing the height shifts your guess and makes you more certain (12 → 9.6 kg). Murphy §3.2.3.

</details>

## 3.4 · Simpson's paradox in one table  
*Section §3.1 · think*

Treatment A cures 81/87 small stones and 192/263 large stones. Treatment B cures 234/270 small and 55/80 large. Which is better within each group? Which looks better overall? Why?

<details><summary><b>Hint</b></summary>

Compute four percentages, then the two overall percentages. Who got the hard cases?

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

Small: A 93% vs B 87%. Large: A 73% vs B 69%. A wins in both groups. Overall: A 273/350 = 78%, B 289/350 = 83%, so B looks better. The reversal happens because A was given most of the hard (large-stone) cases. The group variable confounds the comparison; you have to condition on it (Murphy §3.1.5).

</details>